In [22]:
import pandas as pd
import numpy as np
import random
import math
import time

# Meal Database

In [23]:
import random

# ─── YOUR MEAL DATABASE (from CSV) ───────────────────────────────────────
meals_df = pd.read_csv('data/recipes.csv')
meals_db = []
for _, row in meals_df.iterrows():
    meals_db.append({
        "name": row["Name"],
        "type": row["type"],
        "kcal": row["provided_calories"],
        "cost": row["total_price"],
        "category": row["Category"]
    })

# User Input

In [24]:
# ─── USER INPUT ─────────────────────────────────────────────────────────────
TDEE          = 2000   # target daily calories
DAILY_BUDGET  = 1500    # DZD per day
PREFERENCE    = "Non-Vegetarian"  # or "Vegetarian"
DAYS          = 30

# Domain Building

In [25]:
# ─── STEP 1: BUILD INITIAL DOMAINS ──────────────────────────────────────────
# domains[(day, slot)] = list of valid meals for that slot
# Filter by meal type, user preference, and individual calorie/budget constraints

def build_domains():
    # Define meal slot constraints based on remaining daily calorie allowance
    slot_constraints = {
        "Breakfast": {"calorie_max": TDEE * 0.35, "budget_max": DAILY_BUDGET * 0.35},  # 35% of daily calories/budget
        "Lunch": {"calorie_max": TDEE * 0.45, "budget_max": DAILY_BUDGET * 0.45},      # 45% of daily calories/budget
        "Dinner": {"calorie_max": TDEE * 0.45, "budget_max": DAILY_BUDGET * 0.45}      # 45% of daily calories/budget
    }
    
    domains = {}
    for day in range(DAYS):
        for slot in ["Breakfast", "Lunch", "Dinner"]:
            constraint = slot_constraints[slot]
            domains[(day, slot)] = [
                meal for meal in meals_db
                if meal["type"] == slot
                # User preference filter
                and (PREFERENCE != "Vegetarian" or meal["category"] == "Vegetarian")
                # Individual meal constraints based on remaining daily allowance
                and meal["kcal"] <= constraint["calorie_max"]
                and meal["cost"] <= constraint["budget_max"]
            ]
    return domains

# Constraints

In [26]:
# ─── STEP 2: DEFINE THE CONSTRAINT ──────────────────────────────────────────
# Two meals from different slots of the same day are "compatible" if:
# combining them still leaves room for a valid third meal

def are_compatible(meal_a, meal_b, remaining_slot_min_kcal, remaining_slot_max_kcal):
    """
    Check if meal_a and meal_b can coexist on the same day.
    remaining_slot_min/max = the calorie range of the third slot's cheapest/largest meal.
    """
    combined_kcal = meal_a["kcal"] + meal_b["kcal"]
    combined_cost = meal_a["cost"] + meal_b["cost"]

    low  = TDEE * 0.9   # 1800 kcal
    high = TDEE * 1.1   # 2200 kcal

    # The third meal must bring total into [low, high]
    needed_min = low  - combined_kcal   # third meal needs at least this
    needed_max = high - combined_kcal   # third meal needs at most this

    # Is there overlap between what we need and what's available?
    calorie_ok = needed_max >= remaining_slot_min_kcal and needed_min <= remaining_slot_max_kcal
    budget_ok  = combined_cost < DAILY_BUDGET  # still leaves room for third meal

    return calorie_ok and budget_ok

# AC-3 Algorithm

In [27]:
# ─── STEP 3: AC-3 ───────────────────────────────────────────────────────────

def ac3(domains):
    slots = ["Breakfast", "Lunch", "Dinner"]

    # Queue holds pairs: (day, slot_i, slot_j) meaning
    # "check slot_i against slot_j on this day"
    queue = []
    for day in range(DAYS):
        for s1 in slots:
            for s2 in slots:
                if s1 != s2:
                    queue.append((day, s1, s2))

    while queue:
        day, slot_i, slot_j = queue.pop(0)

        # The "third" slot on this day (not slot_i or slot_j)
        third_slot = [s for s in slots if s != slot_i and s != slot_j][0]
        third_kcals = [m["kcal"] for m in domains[(day, third_slot)]]

        if not third_kcals:
            return False  # No solution possible

        third_min = min(third_kcals)
        third_max = max(third_kcals)


        revised = False

        # For each meal in slot_i, check if ANY meal in slot_j is compatible
        for meal_i in domains[(day, slot_i)][:]:  # copy list so we can delete safely
            compatible_exists = any(
                are_compatible(meal_i, meal_j, third_min, third_max)
                for meal_j in domains[(day, slot_j)]
            )
            if not compatible_exists:
                domains[(day, slot_i)].remove(meal_i)  # prune it
                revised = True

        if revised:
            if len(domains[(day, slot_i)]) == 0:
                return False  # Domain wiped out → no solution

            # Propagate: re-check everything that points TO slot_i
            for slot_k in slots:
                if slot_k != slot_i:
                    queue.append((day, slot_k, slot_i))

    return True  # AC-3 finished, domains are pruned

# Backtracking Search

In [28]:
# ─── STEP 4: BACKTRACKING SEARCH ────────────────────────────────────────────
# Proper backtracking to find a valid assignment for each day

def backtrack_solve(domains, day):
    slots = ["Breakfast", "Lunch", "Dinner"]
    low = TDEE * 0.9
    high = TDEE * 1.1

    def backtrack(index, current_kcal, current_cost, assignment):
        if index == len(slots):
            # Check final constraints
            if low <= current_kcal <= high and current_cost <= DAILY_BUDGET:
                return assignment.copy()
            return None

        slot = slots[index]
        for meal in random.sample(domains[(day, slot)], len(domains[(day, slot)])):
            new_kcal = current_kcal + meal["kcal"]
            new_cost = current_cost + meal["cost"]
            # Early pruning
            if new_cost > DAILY_BUDGET or new_kcal > high:
                continue
            assignment[slot] = meal
            result = backtrack(index + 1, new_kcal, new_cost, assignment)
            if result:
                return result
            del assignment[slot]
        return None

    return backtrack(0, 0, 0, {})

def solve(domains):
    plan = {}
    for day in range(DAYS):
        day_plan = backtrack_solve(domains, day)
        if day_plan is None:
            return None  # No solution for this day
        plan[day] = day_plan
    return plan

# Execution

In [29]:
# ─── RUN IT ─────────────────────────────────────────────────────────────────
print("=== CSP Meal Planning for 30 Days ===")
print(f"Target: {TDEE} kcal/day, Budget: {DAILY_BUDGET} DZD/day, Preference: {PREFERENCE}")
print()

# Time domain building
start_time = time.time()
domains = build_domains()
domain_build_time = time.time() - start_time
print(f"Domain building completed in {domain_build_time:.4f} seconds")
print(f"Domain sizes before AC-3: { {k: len(v) for k, v in domains.items() if k[0] == 0} }")
print()

# Time AC-3
start_time = time.time()
success = ac3(domains)
ac3_time = time.time() - start_time
print(f"AC-3 algorithm completed in {ac3_time:.4f} seconds")
print(f"AC-3 success: {success}")
print(f"Domain sizes after AC-3:  { {k: len(v) for k, v in domains.items() if k[0] == 0} }")
print()

if success:
    # Time solving
    start_time = time.time()
    plan = solve(domains)
    solve_time = time.time() - start_time
    print(f"Backtracking search completed in {solve_time:.4f} seconds")
    
    if plan:
        print("✓ Solution found!")
        print(f"Total computation time: {domain_build_time + ac3_time + solve_time:.4f} seconds")
        print()
        
        # Calculate totals
        total_cost = 0
        total_kcal = 0
        for day in range(DAYS):
            day_cost = sum(meal["cost"] for meal in plan[day].values())
            day_kcal = sum(meal["kcal"] for meal in plan[day].values())
            total_cost += day_cost
            total_kcal += day_kcal
        print(f"Total cost for {DAYS} days: {total_cost:.2f} DZD")
        print(f"Average daily cost: {total_cost/DAYS:.2f} DZD")
        print(f"Total calories: {total_kcal:.0f}")
        print(f"Average daily calories: {total_kcal/DAYS:.2f}")
        
        # Print first 3 days
        print("\nFirst 3 days of the meal plan:")
        for day in range(min(3, DAYS)):
            print(f"\nDay {day+1}:")
            for slot, meal in plan[day].items():
                print(f"  {slot}: {meal['name']} ({meal['kcal']:.0f} kcal, {meal['cost']:.2f} DZD)")
    else:
        print("✗ No solution found after AC-3.")
else:
    print("✗ AC-3 failed: no solution possible.")

print("\n" + "="*50)

=== CSP Meal Planning for 30 Days ===
Target: 2000 kcal/day, Budget: 1500 DZD/day, Preference: Non-Vegetarian

Domain building completed in 0.0030 seconds
Domain sizes before AC-3: {(0, 'Breakfast'): 57, (0, 'Lunch'): 50, (0, 'Dinner'): 51}

AC-3 algorithm completed in 0.0515 seconds
AC-3 success: True
Domain sizes after AC-3:  {(0, 'Breakfast'): 57, (0, 'Lunch'): 49, (0, 'Dinner'): 50}

Backtracking search completed in 0.0240 seconds
✓ Solution found!
Total computation time: 0.0785 seconds

Total cost for 30 days: 21992.15 DZD
Average daily cost: 733.07 DZD
Total calories: 56866
Average daily calories: 1895.53

First 3 days of the meal plan:

Day 1:
  Breakfast: Zucchini Egg Breakfast (250 kcal, 57.30 DZD)
  Lunch: Tajine Zitoun (689 kcal, 328.47 DZD)
  Dinner: Beef Chickpea Tomato Dinner (870 kcal, 522.15 DZD)

Day 2:
  Breakfast: Oat Banana Pancake Bowl (649 kcal, 95.00 DZD)
  Lunch: Hlalem (838 kcal, 265.77 DZD)
  Dinner: Potato Carrot Pea Pan (326 kcal, 48.25 DZD)

Day 3:
  Breakf

# Analysis

In [30]:
# Additional analysis: Check variety and constraints satisfaction
if plan:
    print("\n--- Additional Analysis ---")
    # Check how many unique meals used
    all_meals = set()
    for day_plan in plan.values():
        for meal in day_plan.values():
            all_meals.add(meal['name'])
    print(f"Number of unique meals used: {len(all_meals)}")
    
    # Check constraint satisfaction for each day
    violations = 0
    for day in range(DAYS):
        day_kcal = sum(meal["kcal"] for meal in plan[day].values())
        day_cost = sum(meal["cost"] for meal in plan[day].values())
        if not (TDEE * 0.9 <= day_kcal <= TDEE * 1.1) or day_cost > DAILY_BUDGET:
            violations += 1
    print(f"Days with constraint violations: {violations}")
    
    # Most used meals
    meal_counts = {}
    for day_plan in plan.values():
        for meal in day_plan.values():
            meal_counts[meal['name']] = meal_counts.get(meal['name'], 0) + 1
    most_used = sorted(meal_counts.items(), key=lambda x: x[1], reverse=True)[:5]
    print("Top 5 most used meals:")
    for name, count in most_used:
        print(f"  {name}: {count} times")


--- Additional Analysis ---
Number of unique meals used: 58
Days with constraint violations: 0
Top 5 most used meals:
  Chicken Mchermel: 5 times
  Hlalem: 4 times
  Chorba Vermicelle Beef: 4 times
  Chakhchoukha Sahraoui: 4 times
  Beef Chickpea Tomato Dinner: 3 times
